In [ ]:
import notebooks.data_loader as loader
from importlib import reload
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
import gsw
import mod_argo
import matplotlib.pyplot as plt
from cmocean import cm as cmo


# === CUSTOM FUNCTIONS
import mod_preprocessing as mod_prep
import mod_ocean

# New version of preprocessing feb 3 2026

- add MLD and ADT here before clustering


In [3]:
reload(loader)
[coreDS, coreINDEX] = loader.import_core_data(type = 'L3_only')

In [2]:
[bgcDS_orig, bgcINDEX_orig] = loader.import_bgc_data(type = 'L3_only')

In [ ]:
# Add MLD
reload(mod_prep)
bgcINDEX, bgcDS = mod_prep.add_mixedlayer_pressure(bgcINDEX, bgcDS)
# coreINDEX, coreDS = mod_prep.add_mixedlayer_pressure(coreINDEX, coreDS)

/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(
/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


In [ ]:
folder = '../working-vars/argo/import-L3/'
datetag = '20260121'
# tempINDEX = xr.open_dataset(folder + 'bgcINDEX_validL3_2014-2023_withMLD_acc' + datetag + '.nc')
# tempDS = xr.open_dataset(folder + 'bgcDATA_validL3_2014-2023_withMLD_acc' + datetag + '.nc')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/working-vars/argo/import-L3/bgcDATA_validL3_2014-2023_withMLD_acc20260121.nc'

In [ ]:
# coreDS_ADT = xr.open_dataset('../working-vars/regression/inputs/core/coreDATA_processed_2014-2023_MLD_ADT_SLA_acc20260201.nc')


In [12]:
coreDS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    wmoid        (profid, pressure) float64 1.9e+06 1.9e+06 ... 7.901e+06
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables:
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    mld          (profid) float64 90.55 66.09 13.02 74.86 ... 82.53 71.18 19.52
    mlp          (profid) object 91.27 66.61 13.12 75.46 ... 83.29 71.83 19.69
Attributes:
    title:            Core float profiles with valid surface data (QC flags 1...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode; GDAC snapshot Jan 2025
    date:             20250424

In [11]:
coreINDEX

<xarray.Dataset>
Dimensions:      (profid: 328936)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
Data variables: (12/17)
    pressure     (profid) float64 ...
    wmoid        (profid) float64 ...
    latitude     (profid) float64 ...
    longitude    (profid) float64 ...
    datetime     (profid) object ...
    yearday      (profid) float64 ...
    ...           ...
    salinity     (profid) float64 ...
    time_qc      (profid) float64 ...
    position_qc  (profid) float64 ...
    bathymetry   (profid) float64 ...
    mld          (profid) float64 90.55 66.09 13.02 74.86 ... 82.53 71.18 19.52
    mlp          (profid) object 91.27 66.61 13.12 75.46 ... 83.29 71.83 19.69
Attributes:
    title:    Core profiles with valid QC data, masked to open ocean bathymet...

In [ ]:
socatDF = mod_prep.convert_socat_fco2(socatDS)

# Add SSH and wind products as features (originally from 2.0_RUN)

In [ ]:
# Use save columns
# floatDF = floatDF_final.copy()
# shipDF = shipDF_final.copy()

# Import daily satellite ADT at 1/4 deg resolution
adt_dict = {k:None for k in range(2014,2024)}
for open_yr in adt_dict.keys():
    folder = '/Volumes/crusoe-repo/data/copernicusmarine/' # used to be in OneDrive/Code/CRUSOE
    filepath = ('cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt-sla_179.94W-179.94E_88.94S-35.06S_' + str(open_yr) + '-01-01-' + str(open_yr) + '-12-31.nc')
    adt_dict[open_yr] = xr.open_dataset(folder + filepath)



In [ ]:
adt_dict

{2014: <xarray.Dataset>
 Dimensions:    (time: 365, latitude: 432, longitude: 2880)
 Coordinates:
   * latitude   (latitude) float32 -88.94 -88.81 -88.69 ... -35.31 -35.19 -35.06
   * longitude  (longitude) float32 -179.9 -179.8 -179.7 ... 179.7 179.8 179.9
   * time       (time) datetime64[ns] 2014-01-01 2014-01-02 ... 2014-12-31
 Data variables:
     adt        (time, latitude, longitude) float64 ...
     sla        (time, latitude, longitude) float64 ...
 Attributes:
     Conventions:               CF-1.6
     title:                     DT merged all satellites Global Ocean Gridded ...
     history:                   2024-10-23 12:55:06Z: Creation
     source:                    Altimetry measurements
     institution:               CLS, CNES
     contact:                   servicedesk.cmems@mercator-ocean.eu
     comment:                   Sea Surface Height measured by Altimetry and d...
     references:                http://marine.copernicus.eu
     copernicusmarine_version:  2.

In [ ]:
# Run for all ten years
allplat =  pd.concat([floatDF_processed, shipDF_processed], axis=0)
allplat =  mod_ocean.expand_datetime(allplat, type='dataframe')
allplat_annual = {k:None for k in range(2014,2024)} # store by year

def get_adt_for_obs(row):
    return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').adt.values.tolist()
def get_sla_for_obs(row):
    return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').sla.values.tolist()

for yr in range(2014,2024):
    allplat_annual[yr] = allplat[allplat.year==yr]
    adt_data = adt_dict[yr]
    #  allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest'), axis=1)
    allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: get_adt_for_obs(row), axis=1)
    allplat_annual[yr]['sla'] =  allplat_annual[yr].apply(lambda row: get_sla_for_obs(row), axis=1)

# Separate into ship and float
allplat_added = pd.concat(allplat_annual.values(), axis=0)
shipDF_added = allplat_added[allplat_added.wmoid.isna()]

floatDF_added = allplat_added[~allplat_added.wmoid.isna()]
floatDF_added['wmoid'] = floatDF_added['wmoid'].astype(int)

/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_91931/2347411604.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: get_adt_for_obs(row), axis=1)
/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_91931/2347411604.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allplat_annual[yr]['sla'] =  allplat_annual[yr].apply(lambda row: get_sla_for_obs(row), axis=1)
/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_919

In [ ]:
# === Optional save: 
save = False
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/' + pcm_params + '/'

    # Floats
    filename = 'floatDF_ADT_SLA_soccom20m_pCO2_pHbias5_pK1_PCM_' + pcm_params + '_acc' + datetag + '.csv'
    use_cols = ['profid', 'wmoid', 'prof_datetag', 'cluster', 
              'datetime', 'latitude', 'longitude', 
              'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
              'CT', 'SA', 'mld', 
              'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
              'pco2', 'pco2_atm', 'delta_pco2',
              'year', 'month', 'day', 
              'adt', 'sla']
    floatDF_added[use_cols].to_csv(filepath + filename)
    print('Saved floatDF to ' + filepath + filename)

    # Ship data
    filename = 'shipDF_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_' + pcm_params + '_acc' + datetag + '.csv'
    use_cols = ['sampleid', 'cruiseid', 'nearest_profid', 
              'cluster', 'datetime', 'latitude', 'longitude', 
              'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
              'CT', 'SA', 'mld', 
              'pco2', 'pco2_atm', 'delta_pco2', 
              'atm_pres_hPa', 'fco2rec', 
              'year', 'month', 'day', 
              'adt','sla']
    shipDF_added[use_cols].to_csv(filepath + filename)
    print('Saved shipDF to ' + filepath + filename)



Saved floatDF to ../working-vars/regression/inputs/pc8_gmm6/floatDF_ADT_SLA_soccom20m_pCO2_pHbias5_pK1_PCM_pc8_gmm6_acc20260123.csv
Saved shipDF to ../working-vars/regression/inputs/pc8_gmm6/shipDF_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_pc8_gmm6_acc20260123.csv


# Add ADT/SLA to Core Argo


In [ ]:
# coreDF = coreDS.to_dataframe().reset_index().copy()

# coreDF['linear_time'] = coreDF.datetime.apply(lambda x: mod_ocean.datetime2ytd(np.datetime64(x), ref_time='2014-01-01'))
# coreDF['ydcos']= mod_ocean.get_ydsines(coreDF.linear_time.values)[0]
# coreDF['ydsin']= mod_ocean.get_ydsines(coreDF.linear_time.values)[1]
# coreDF = mod_ocean.add_decimalyr(coreDF)


# reload(mod_prep)
# mod_prep.add_regression_vars(coreDF, timeOnly=True)

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [ ]:
coreDF = coreDS.to_dataframe().reset_index().copy()

coreDF['linear_time'] = coreDF.datetime.apply(lambda x: mod_ocean.datetime2ytd(np.datetime64(x), ref_time='2014-01-01'))
coreDF['ydcos']= mod_ocean.get_ydsines(coreDF.linear_time.values)[0]
coreDF['ydsin']= mod_ocean.get_ydsines(coreDF.linear_time.values)[1]
coreDF = mod_ocean.add_decimalyr(coreDF)

coreDF =  mod_ocean.expand_datetime(coreDF, type='dataframe')



In [ ]:
from tqdm import tqdm

In [ ]:
# ==== COMMENT OUT TO KEEP OLD PROGRESS
# coreDF_annual = {k:None for k in range(2014,2024)} # store by year


# === Optional save: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 
filepath = '../working-vars/regression/inputs/core/'


# Run over all ten years
for yr in tqdm(range(2015,2024)):
    print('==> Processing year ' + str(yr))
    adt_data = adt_dict[yr]
    core_data = coreDF[coreDF.year==yr].copy()
    core_data['adt'] = np.tile(np.nan, len(core_data))
    core_data['sla'] = np.tile(np.nan, len(core_data))

    for ind, row in core_data.iterrows():
        closest_row = adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest')
        adtval = closest_row.adt.values.tolist()
        slaval = closest_row.sla.values.tolist()
        core_data.loc[ind, 'adt'] = adtval
        core_data.loc[ind, 'sla'] = slaval
   
    coreDF_annual[yr] = core_data

    if save:
        filename = 'coreDF_yr' + str(yr)+ '_ADT_SLA_acc' + datetag + '.csv'
        coreDF_annual[yr].to_csv(filepath + filename)
        print('Saved year ' + str(yr) + ' to ' + filename)



coreDF_added = pd.concat(coreDF_annual.values(), axis=0)

  0%|          | 0/9 [00:00<?, ?it/s]

==> Processing year 2015


 11%|█         | 1/9 [1:26:06<11:28:51, 5166.46s/it]

Saved year 2015 to coreDF_yr2015_ADT_SLA_acc20260130.csv
==> Processing year 2016


 22%|██▏       | 2/9 [2:58:19<10:27:53, 5381.94s/it]

...

Saved year 2021 to coreDF_yr2021_ADT_SLA_acc20260130.csv
==> Processing year 2022


 89%|████████▉ | 8/9 [11:36:27<1:23:26, 5006.55s/it]

Saved year 2022 to coreDF_yr2022_ADT_SLA_acc20260130.csv
==> Processing year 2023


100%|██████████| 9/9 [13:07:21<00:00, 5249.09s/it]  

Saved year 2023 to coreDF_yr2023_ADT_SLA_acc20260130.csv


In [ ]:
# === Optional save of full DF : 
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/core/'
    # Floats
    filename = 'coreDF_yrs2014-2023_ADT_SLA_processed_acc' + datetag + '.csv'
    coreDF_added.to_csv(filepath + filename)
    print('Saved coreDF to ' + filepath + filename)

Saved coreDF to ../working-vars/regression/inputs/core/coreDF_yrs2014-2023_ADT_SLA_processed_acc20260131.csv


In [ ]:
coreDF_added.columns

Index(['profid', 'pressure', 'CT', 'SA', 'sigma0', 'spice', 'temperature',
       'salinity', 'yearday', 'latitude', 'longitude', 'wmoid', 'datetime',
       'mld', 'mlp', 'linear_time', 'ydcos', 'ydsin', 'decimalyr', 'year',
       'month', 'day', 'adt', 'sla'],
      dtype='object')

In [ ]:
reload(mod_prep)
coreDF_added

In [ ]:
coreDS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    wmoid        (profid, pressure) float64 1.9e+06 1.9e+06 ... 7.901e+06
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables:
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    mld          (profid) float64 90.55 66.09 13.02 74.86 ... 82.53 71.18 19.52
    mlp          (profid) object 91.27 66.61 13.12 75.46 ... 83.29 71.83 19.69
Attributes:
    title:            Core float profiles with valid surface data (QC flags 1...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode; GDAC snapshot Jan 2025
    date:             20250424

In [ ]:
coreDS

In [ ]:
coreDS_added = mod_argo.to_xr_dataset(coreDF_added)

In [ ]:
temp = xr.Dataset.from_dataframe(coreDF_added.set_index(["profid", 'pressure']))
nc_attrs={'date':str(datetime.now()),
          'adt_type': "Global Ocean Gridded L 4 Sea Surface Heights And Derived Variables Reprocessed 1993 Ongoing",
          'adt_source': "https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L4_MY_008_047/description"}

argo_DS = temp.set_coords(['profid', 'datetime', 'yearday', 'latitude', 'longitude'])
argo_DS = argo_DS.assign_attrs(nc_attrs)
argo_DS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables: (12/18)
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    ...           ...
    decimalyr    (profid, pressure) float64 2.014e+03 2.014e+03 ... 2.024e+03
    year         (profid, pressure) int64 2014 2014 2014 2014 ... 2023 2023 2023
    month        (profid, pressure) int64 1 1 1 1 1 1 1 ... 12 12 12 12 12 12 12
    day          (profid, pressure) int64 2 2 2 2 2 2 2 ... 23 23 23 23 23 23 23
    adt          (profid, pressure) float64 0.5323 0.5323 ... -0.5156 -0.5156
    sla          (profid, pressure) float64 -0.012 -0.012 ... -0.0052 -0.0052
Attributes:
    date:        2026-02-01 12:29:20.321568
    adt_type:    Global Ocean Gridded L 4 Sea Surface Heights And Derived Var...
    adt_source:  https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L...

In [ ]:
# === Optional save of full xr Dataset: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/core/'
    # Floats
    filename = 'coreDATA_processed_2014-2023_MLD_ADT_SLA_acc' + datetag + '.nc'
    argo_DS.to_netcdf(filepath + filename)
    print('Saved coreDS to ' + filepath + filename)

    # later accessed through loader.import_core_data(L3_only = False)

Saved coreDS to ../working-vars/regression/inputs/core/coreDS_yrs2014-2023_ADT_SLA_processed_acc20260201.nc


In [ ]:
argo_DS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables: (12/18)
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    ...           ...
    decimalyr    (profid, pressure) float64 2.014e+03 2.014e+03 ... 2.024e+03
    year         (profid, pressure) int64 2014 2014 2014 2014 ... 2023 2023 2023
    month        (profid, pressure) int64 1 1 1 1 1 1 1 ... 12 12 12 12 12 12 12
    day          (profid, pressure) int64 2 2 2 2 2 2 2 ... 23 23 23 23 23 23 23
    adt          (profid, pressure) float64 0.5323 0.5323 ... -0.5156 -0.5156
    sla          (profid, pressure) float64 -0.012 -0.012 ... -0.0052 -0.0052
Attributes:
    date:        2026-02-01 12:29:20.321568
    adt_type:    Global Ocean Gridded L 4 Sea Surface Heights And Derived Var...
    adt_source:  https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L...